In [1]:
import pandas as pd
dataset=pd.read_csv("Social_Network_Ads.csv")
dataset
dataset=pd.get_dummies(dataset,drop_first=True)
dataset=dataset.drop(columns=["User ID"])
dataset
x=dataset[["Age","EstimatedSalary","Gender_Male"]]
y=dataset[["Purchased"]]
print(y)
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=1/8,random_state=52)
dataset[["Purchased"]].value_counts()
dataset.shape

     Purchased
0            0
1            0
2            0
3            0
4            0
..         ...
395          1
396          1
397          1
398          0
399          1

[400 rows x 1 columns]


(400, 4)

In [2]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, r2_score

pipe = Pipeline([
    ('discretizer', KBinsDiscretizer(encode='ordinal', strategy='uniform')),
    ('classifier', CategoricalNB())
])

param_grid = {
    'discretizer__n_bins': [3, 5, 7, 10],
    'classifier__alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    'classifier__fit_prior': [True, False],
    'classifier__min_categories': [None, 10, 20]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='accuracy',
    refit=True,
    verbose=3,
    n_jobs=-1
)

grid.fit(x_train, y_train.values.ravel())

print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)

y_pred = grid.best_estimator_.predict(x_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("R_score:", r2_score(y_test, y_pred))

Fitting 5 folds for each of 144 candidates, totalling 720 fits
Best Parameters: {'classifier__alpha': 0.01, 'classifier__fit_prior': True, 'classifier__min_categories': None, 'discretizer__n_bins': 5}
Best Accuracy: 0.917142857142857
Test Accuracy: 0.9
R_score: 0.5404411764705882


In [3]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[32  2]
 [ 3 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.93        34
           1       0.87      0.81      0.84        16

    accuracy                           0.90        50
   macro avg       0.89      0.88      0.88        50
weighted avg       0.90      0.90      0.90        50



In [4]:
re=grid.cv_results_ 
grid_predictions = grid.predict(x_test)
from sklearn.metrics import r2_score
r_score=r2_score(y_test,grid_predictions)
print("The R_score value for best parameter {}:".format(grid.best_params_),r_score)

The R_score value for best parameter {'classifier__alpha': 0.01, 'classifier__fit_prior': True, 'classifier__min_categories': None, 'discretizer__n_bins': 5}: 0.5404411764705882


In [5]:
table=pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__alpha,param_classifier__fit_prior,param_classifier__min_categories,param_discretizer__n_bins,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.156782,0.011886,0.007528,0.000682,0.01,True,None,3,"{'classifier__alpha': 0.01, 'classifier__fit_p...",0.871429,0.857143,0.900000,0.900000,0.857143,0.877143,0.019378,110
1,0.163543,0.020035,0.008419,0.000822,0.01,True,None,5,"{'classifier__alpha': 0.01, 'classifier__fit_p...",0.914286,0.914286,0.928571,0.942857,0.885714,0.917143,0.018952,1
2,0.077193,0.074549,0.007293,0.001484,0.01,True,None,7,"{'classifier__alpha': 0.01, 'classifier__fit_p...",0.928571,0.928571,0.928571,0.914286,0.828571,0.905714,0.038966,48
3,0.013613,0.001136,0.006231,0.000618,0.01,True,None,10,"{'classifier__alpha': 0.01, 'classifier__fit_p...",0.900000,0.900000,0.914286,0.914286,0.871429,0.900000,0.015649,59
4,0.013176,0.000639,0.005912,0.000709,0.01,True,10,3,"{'classifier__alpha': 0.01, 'classifier__fit_p...",0.871429,0.857143,0.900000,0.900000,0.857143,0.877143,0.019378,110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,0.018028,0.003063,0.009919,0.001566,5.00,False,10,10,"{'classifier__alpha': 5.0, 'classifier__fit_pr...",0.914286,0.885714,0.914286,0.900000,0.871429,0.897143,0.016660,84
140,0.030663,0.004079,0.015416,0.001752,5.00,False,20,3,"{'classifier__alpha': 5.0, 'classifier__fit_pr...",0.871429,0.857143,0.900000,0.900000,0.857143,0.877143,0.019378,110
141,0.038303,0.009756,0.017004,0.004857,5.00,False,20,5,"{'classifier__alpha': 5.0, 'classifier__fit_pr...",0.914286,0.914286,0.928571,0.928571,0.885714,0.914286,0.015649,11
142,0.035685,0.007898,0.014899,0.002503,5.00,False,20,7,"{'classifier__alpha': 5.0, 'classifier__fit_pr...",0.900000,0.885714,0.928571,0.900000,0.828571,0.888571,0.033074,98
